# SQL for Data Science: The Complete Guide
This notebook demonstrates how to interact with relational databases using Python's `sqlite3` library and `pandas`. We will build a complete, functional database from scratch, covering everything from table creation to complex joins and subqueries.

In [1]:
import sqlite3
import pandas as pd

# 1. Databases & Installation (Connection)
# This will create a new file named 'gym_management_demo.db' in your folder
conn = sqlite3.connect('gym_management_demo.db')
cursor = conn.cursor()

print("✅ Connected to SQLite Database: gym_management_demo.db")

✅ Connected to SQLite Database: gym_management_demo.db


## 1. Creating Tables, Constraints & Foreign Keys
A relational database uses **Primary Keys** (unique IDs) and **Foreign Keys** (links) to maintain data integrity. Constraints like `NOT NULL` and `UNIQUE` prevent bad data from entering the system.

In [2]:
# Create Parent Table: Membership Plans
cursor.execute('''
CREATE TABLE IF NOT EXISTS membership_plans (
    plan_id INTEGER PRIMARY KEY,
    plan_name TEXT UNIQUE NOT NULL,
    monthly_fee REAL NOT NULL
);
''')

# Create Child Table: Gym Members
# Notice the FOREIGN KEY linking a member to a specific plan
cursor.execute('''
CREATE TABLE IF NOT EXISTS members (
    member_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER CHECK(age >= 16),
    plan_id INTEGER,
    join_date DATE DEFAULT CURRENT_DATE,
    FOREIGN KEY (plan_id) REFERENCES membership_plans(plan_id)
);
''')

conn.commit()
print("Tables 'membership_plans' and 'members' created successfully!")

Tables 'membership_plans' and 'members' created successfully!


## 2. CRUD Operations & Transactions
* **C**reate (`INSERT`)
* **R**ead (`SELECT`)
* **U**pdate (`UPDATE`)
* **D**elete (`DELETE`)

**Transactions** ensure that a block of code executes entirely or not at all. If an error occurs midway, we `ROLLBACK` to the previous state.

In [3]:
try:
    # INSERT: Add Plans
    cursor.execute("INSERT OR IGNORE INTO membership_plans (plan_id, plan_name, monthly_fee) VALUES (1, 'Basic', 1500), (2, 'Premium', 2500), (3, 'Elite', 4000)")
    
    # INSERT: Add Members
    cursor.execute("INSERT INTO members (name, age, plan_id) VALUES ('Amit', 24, 1), ('Neha', 28, 2), ('Rahul', 22, 1), ('Zoya', 30, 3)")
    
    # UPDATE: Rahul upgrades his plan
    cursor.execute("UPDATE members SET plan_id = 2 WHERE name = 'Rahul'")
    
    # DELETE: A member cancels their membership
    cursor.execute("DELETE FROM members WHERE name = 'Amit'")
    
    # COMMIT: Save all these changes permanently
    conn.commit()
    print("✅ Transaction committed successfully!")
    
except Exception as e:
    # ROLLBACK: Undo everything if something failed
    conn.rollback()
    print(f"❌ Transaction failed. Rolled back. Error: {e}")

# Verify the current state of members
display(pd.read_sql("SELECT * FROM members", conn))

✅ Transaction committed successfully!


,member_id,name,age,plan_id,join_date
0,2,Neha,28,2,2026-03-26
1,3,Rahul,22,2,2026-03-26
2,4,Zoya,30,3,2026-03-26


## 3. Joins & UNION


[Image of SQL Joins Venn Diagram]

* **INNER JOIN:** Returns records that have matching values in both tables.
* **LEFT JOIN:** Returns all records from the left table, and the matched records from the right table.
* **UNION:** Combines the result-set of two or more SELECT statements vertically.

In [4]:
# INNER JOIN: See exactly which plan each active member is on and how much they pay
join_query = """
SELECT m.member_id, m.name, p.plan_name, p.monthly_fee
FROM members m
INNER JOIN membership_plans p ON m.plan_id = p.plan_id;
"""
print("--- Member Plan Details ---")
display(pd.read_sql(join_query, conn))

# UNION: Create a combined directory of all distinct text names in our system
union_query = """
SELECT name AS Directory_Entry, 'Member' AS Category FROM members
UNION
SELECT plan_name AS Directory_Entry, 'Subscription Plan' AS Category FROM membership_plans;
"""
print("\n--- System Directory (UNION) ---")
display(pd.read_sql(union_query, conn))

--- Member Plan Details ---


,member_id,name,plan_name,monthly_fee
0,2,Neha,Premium,2500.0
1,3,Rahul,Premium,2500.0
2,4,Zoya,Elite,4000.0



--- System Directory (UNION) ---


,Directory_Entry,Category
0,Basic,Subscription Plan
1,Elite,Subscription Plan
2,Neha,Member
3,Premium,Subscription Plan
4,Rahul,Member
5,Zoya,Member


## 4. Aggregation (GROUP BY) & Subqueries
* **GROUP BY:** Groups rows sharing a property so you can apply aggregate functions like `COUNT()`, `SUM()`, or `AVG()`.
* **Subqueries:** Queries nested inside another query, usually in the `WHERE` clause.

In [5]:
# Let's add two more members to make the grouping more interesting
cursor.execute("INSERT INTO members (name, age, plan_id) VALUES ('Priya', 26, 2), ('Vikram', 32, 2)")
conn.commit()

# GROUP BY: Calculate total expected monthly revenue per plan
groupby_query = """
SELECT p.plan_name, COUNT(m.member_id) as total_subscribers, SUM(p.monthly_fee) as projected_revenue
FROM membership_plans p
LEFT JOIN members m ON p.plan_id = m.plan_id
GROUP BY p.plan_name;
"""
print("--- Revenue by Plan ---")
display(pd.read_sql(groupby_query, conn))

# SUBQUERY: Find members who pay more than the average monthly fee across all plans
subquery = """
SELECT m.name, p.plan_name, p.monthly_fee
FROM members m
JOIN membership_plans p ON m.plan_id = p.plan_id
WHERE p.monthly_fee > (SELECT AVG(monthly_fee) FROM membership_plans);
"""
print("\n--- Premium/Elite Members (Subquery) ---")
display(pd.read_sql(subquery, conn))

--- Revenue by Plan ---


,plan_name,total_subscribers,projected_revenue
0,Basic,0,1500.0
1,Elite,1,4000.0
2,Premium,4,10000.0



--- Premium/Elite Members (Subquery) ---


,name,plan_name,monthly_fee
0,Zoya,Elite,4000.0


## 5. Views, Indexes & Python Stored Procedures
* **Views:** Virtual tables that save complex queries for easy access later.
* **Indexes:** Background data structures that significantly speed up `SELECT` queries on large tables.
* **Stored Procedures:** Reusable SQL code blocks. Since SQLite doesn't natively support full stored procedures, we wrap parameter-based SQL in a Python function.

In [6]:
# 1. CREATE VIEW: Save the revenue query as a virtual table
cursor.execute("""
CREATE VIEW IF NOT EXISTS Revenue_Report AS
SELECT p.plan_name, COUNT(m.member_id) as total_subscribers
FROM membership_plans p
LEFT JOIN members m ON p.plan_id = m.plan_id
GROUP BY p.plan_name;
""")
print("--- Querying the View ---")
display(pd.read_sql("SELECT * FROM Revenue_Report", conn))

# 2. CREATE INDEX: Speed up searching for members by name in huge databases
cursor.execute("CREATE INDEX IF NOT EXISTS idx_member_name ON members(name);")
print("\n✅ Index 'idx_member_name' created.")

# 3. Python "Stored Procedure": Safely insert data using parameters
def register_new_member(connection, name, age, plan_id):
    """Safely executes an INSERT statement using parameterized queries to prevent SQL Injection."""
    sql = "INSERT INTO members (name, age, plan_id) VALUES (?, ?, ?)"
    cur = connection.cursor()
    cur.execute(sql, (name, age, plan_id))
    connection.commit()
    print(f"Stored Procedure Success: Registered '{name}' to plan ID {plan_id}.")

register_new_member(conn, 'Sana', 23, 3)

# Always close your connection when the notebook is done!
conn.close()

--- Querying the View ---


,plan_name,total_subscribers
0,Basic,0
1,Elite,1
2,Premium,4



✅ Index 'idx_member_name' created.
Stored Procedure Success: Registered 'Sana' to plan ID 3.
